# Chapter 13 &mdash; A DTM for $w\#w$, an NDTM for $ww$

**Concept 9 of the Chapter 13 decomposition:** *A DTM for $w\#w$, an NDTM for $ww$, and Why Nondeterminism Adds No Power*

The separator makes $w\#w$ deterministic; $ww$ needs a guessed midpoint &mdash; but a DTM can find it too.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-DTM-For-wsw-And-NDTM-For-ww/Concept-DTM-For-wsw-And-NDTM-For-ww.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.AnimateTM      import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateTM as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateTM, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


$w\#w$ and $ww$ differ by one symbol, and that symbol changes the design completely.

**$w\#w$ is deterministic.** The `#` tells you where the halves meet: cross off the
leftmost unmarked symbol on the left, walk past the `#`, check and cross off the
matching one on the right, return, repeat.

**$ww$ has no separator.** The natural machine **guesses** the midpoint &mdash; a
nondeterministic TM, as in Chapter 12's palindrome PDA.

But unlike PDA, **nondeterminism adds no power to TMs.** A deterministic TM can simply
*try every midpoint in turn* &mdash; it has the tape to keep track. The cost is time, not
computability, and that distinction (possible vs. efficient) is where complexity theory
begins.

## 2. Definitions

### Crossing-off, simulated so the algorithm is visible

In [ ]:
def wsw_algorithm(tape, trace=False):
    # the deterministic cross-off algorithm for w#w
    if tape.count('#') != 1: return False
    s = list(tape)
    i = 0
    while i < len(s) and s[i] != '#':
        hashpos = s.index('#')
        j = hashpos + 1 + i
        if j >= len(s) or s[j] != s[i]: return False
        if trace:
            t = s[:]; t[i] = t[i].upper(); t[j] = t[j].upper()
            print("   ", ''.join(t))
        i += 1
    return len(s) == 2 * i + 1

def ww_deterministic(tape):
    # a DTM can TRY every midpoint -- no guessing needed, just more work
    n = len(tape)
    for mid in range(n + 1):
        if tape[:mid] == tape[mid:]:
            return True, mid
    return False, None

### The nondeterministic guess, as a Jove TM

In [ ]:
GuessTM = md2mc('''TM
!! A tiny illustration of GUESSING: at any point the machine may decide
!! "the midpoint is here" and switch to the checking phase.
I : 0 ; 0 , R -> I
I : 1 ; 1 , R -> I
I : 0 ; 0 , R -> C      !! guess: the midpoint is just after this cell
I : 1 ; 1 , R -> C
C : 0 ; 0 , R -> C
C : 1 ; 1 , R -> C
C : . ; . , S -> F
''')

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch13&nbsp;8.&nbsp;Simple TM Examples: Bit Flipper and "Contains 101"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-Simple-TM-Examples/Concept-Simple-TM-Examples.ipynb) &nbsp;&middot;&nbsp; [**Chapter 13** index](https://github.com/ganeshutah/Jove/blob/master/Chapter13-TM/README.md) &nbsp;&middot;&nbsp; [Ch13&nbsp;10.&nbsp;A TM for the Collatz ($3x+1$) Problem: Termination as an Open Question](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-Collatz-TM/Concept-Collatz-TM.ipynb)&nbsp;&rarr;

---

## 3. Tests

The $w\#w$ algorithm, crossing off in lock-step.

In [ ]:
print("checking 'abb#abb' :")
ok = wsw_algorithm('abb#abb', trace=True)
print("  ->", ok)
assert ok
print()
print("checking 'abb#aba' :", wsw_algorithm('abb#aba'))
assert not wsw_algorithm('abb#aba')

It is fully deterministic: the `#` removes every choice.

In [ ]:
from itertools import product
cases = [(''.join(a) + '#' + ''.join(b))
         for k in range(3) for a in product('ab', repeat=k)
         for j in range(3) for b in product('ab', repeat=j)]
bad = [c for c in cases
       if wsw_algorithm(c) != (c.split('#')[0] == c.split('#')[1])]
print("mismatches over %d cases :" % len(cases), bad)
assert not bad

**$ww$ has no separator**, so the midpoint must be found.

In [ ]:
for t in ['abab', 'abba', 'aa', 'aba', '']:
    ok, mid = ww_deterministic(t)
    print("  %-7r is ww? %-6s midpoint %s" % (t, ok, mid))
assert ww_deterministic('abab')[0] and not ww_deterministic('abba')[0]

A nondeterministic machine **guesses**; Jove explores all the guesses at once.

In [ ]:
nd = [(k, sorted(v)) for k, v in GuessTM["Delta"].items() if len(v) > 1]
print("nondeterministic entries :", len(nd))
assert nd
print("accepts '0101' ?", tm_accepts(GuessTM, '0101', fuel=80))

**But a DTM can try every midpoint** &mdash; nondeterminism costs time, not power.

In [ ]:
import itertools
def ww_dtm_cost(t):
    tries = 0
    for mid in range(len(t) + 1):
        tries += 1
        if t[:mid] == t[mid:]: return True, tries
    return False, tries

for t in ['abab', 'aabbaabb', 'abcabc']:
    ok, tries = ww_dtm_cost(t)
    print("  %-10r ww? %-6s midpoints tried %d" % (t, ok, tries))
print("\nn+1 attempts instead of one lucky guess.  Slower, not weaker.")
print("(Contrast Chapter 12: a DPDA genuinely CANNOT do the palindrome language.)")

## 4. Animation

The guessing machine: the fork out of `I` is the midpoint guess.

In [ ]:
from jove.AnimateTM import *
AnimateTM(GuessTM, FuseEdges=True)

## 5. Exercises


1. Write the $w\#w$ machine out as Jove TM markdown. How many states?
2. Why can a DTM simulate an NDTM but a DPDA cannot simulate an NPDA?
3. What is the time cost of the general NDTM-to-DTM simulation?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter13-TM/Concept-DTM-For-wsw-And-NDTM-For-ww')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')